In [1]:
import pandas as pd
import re
from transformers import pipeline, AutoConfig

In [2]:
df = pd.read_csv("Tweets.csv")
print(df.shape)
print(df.columns.tolist())
print(df['airline_sentiment'].value_counts())

(14640, 15)
['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


In [3]:
df = df[['text', 'airline', 'airline_sentiment', 'tweet_created']]
df = df.dropna()
print(df.shape)

(14640, 4)


In [4]:
def preprocess(text):
    tokens = []
    for token in text.split():
        if token.startswith('@') and len(token) > 1:
            tokens.append('@user')
        elif token.startswith('http'):
            tokens.append('http')
        else:
            tokens.append(token)
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

# Verify it's working correctly
print(df['clean_text'].head(3))

0                               @user What @user said.
1    @user plus you've added commercials to the exp...
2    @user I didn't today... Must mean I need to ta...
Name: clean_text, dtype: object


In [6]:
import os
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import pipeline

MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=MODEL,
    tokenizer=MODEL,
    return_all_scores=False
)

test_texts = df['clean_text'].head(5).tolist()
test_results = sentiment_pipeline(test_texts, truncation=True, max_length=128)

for text, result in zip(test_texts, test_results):
    print(f"Label: {result['label']} | Score: {result['score']:.2f} | Text: {text[:60]}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     | Details
--------------------------------+------------+--------
roberta.pooler.dense.bias       | UNEXPECTED |        
roberta.pooler.dense.weight     | UNEXPECTED |        
roberta.embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label: neutral | Score: 0.94 | Text: @user What @user said.
Label: negative | Score: 0.89 | Text: @user plus you've added commercials to the experience... tac
Label: neutral | Score: 0.67 | Text: @user I didn't today... Must mean I need to take another tri
Label: negative | Score: 0.89 | Text: @user it's really aggressive to blast obnoxious "entertainme
Label: negative | Score: 0.94 | Text: @user and it's a really big bad thing about it


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

In [8]:
all_results = sentiment_pipeline(
    df['clean_text'].tolist(),
    truncation=True,
    max_length=128,
    batch_size=16
)

df['predicted_sentiment'] = [r['label'] for r in all_results]
df['sentiment_score'] = [round(r['score'], 3) for r in all_results]

print(df['predicted_sentiment'].value_counts())

predicted_sentiment
negative    7510
neutral     3980
positive    3150
Name: count, dtype: int64


In [9]:
df.to_csv("tweets_with_sentiment.csv", index=False)
print("Saved. Shape:", df.shape)
print(df[['airline', 'clean_text', 'predicted_sentiment', 'sentiment_score']].head(5))

Saved. Shape: (14640, 7)
          airline                                         clean_text  \
0  Virgin America                             @user What @user said.   
1  Virgin America  @user plus you've added commercials to the exp...   
2  Virgin America  @user I didn't today... Must mean I need to ta...   
3  Virgin America  @user it's really aggressive to blast obnoxiou...   
4  Virgin America     @user and it's a really big bad thing about it   

  predicted_sentiment  sentiment_score  
0             neutral            0.940  
1            negative            0.888  
2             neutral            0.667  
3            negative            0.888  
4            negative            0.935  


In [10]:
label_map = {
    'positive': 'positive',
    'neutral': 'neutral', 
    'negative': 'negative'
}

df['original_lower'] = df['airline_sentiment'].str.lower()
df['model_lower'] = df['predicted_sentiment'].str.lower()

agreement = (df['original_lower'] == df['model_lower']).mean()
print(f"Model agrees with original labels: {agreement:.1%}")
print(f"Disagreement rate: {1 - agreement:.1%}")
print(df[df['original_lower'] != df['model_lower']][['text', 'airline_sentiment', 'predicted_sentiment']].head(10))

Model agrees with original labels: 76.9%
Disagreement rate: 23.1%
                                                 text airline_sentiment  \
1   @VirginAmerica plus you've added commercials t...          positive   
6   @VirginAmerica yes, nearly every time I fly VX...          positive   
7   @VirginAmerica Really missed a prime opportuni...           neutral   
10  @VirginAmerica did you know that suicide is th...           neutral   
15      @VirginAmerica SFO-PDX schedule is still MIA.          negative   
27  @VirginAmerica do you miss me? Don't worry we'...           neutral   
29  @VirginAmerica LAX to EWR - Middle seat on a r...           neutral   
35  Nice RT @VirginAmerica: Vibe with the moodligh...           neutral   
40  @VirginAmerica View of downtown Los Angeles, t...          positive   
41  @VirginAmerica Hey, first time flyer next week...          negative   

   predicted_sentiment  
1             negative  
6             negative  
7             negative  
10      

In [11]:
examples = df[df['original_lower'] != df['model_lower']][['text', 'airline_sentiment', 'predicted_sentiment']].head(3)

output = f"""SENTIMENT MODEL — DISAGREEMENT ANALYSIS
========================================

Model agrees with original labels: 76.9%
Disagreement rate: 23.1%

TOP 3 DISAGREEMENT EXAMPLES
-----------------------------
"""

for i, (_, row) in enumerate(examples.iterrows(), 1):
    output += f"""
Example {i}:
  Tweet:          {row['text']}
  Original label: {row['airline_sentiment']}
  Model label:    {row['predicted_sentiment']}
"""

with open("disagreement_analysis.txt", "w", encoding="utf-8") as f:
    f.write(output)

print("Saved to disagreement_analysis.txt")

Saved to disagreement_analysis.txt
